# Module 06 — Chains (LCEL)

Until now you did two steps by hand every time: fill a prompt template, then invoke the model, then read `.text`. **LCEL** (LangChain Expression Language) lets you connect those steps with the `|` operator into a single **chain**.

```
prompt | model | parser
```

Read `|` as "pipe the output of the left into the right." The reward: the whole chain supports `invoke`, `stream`, and `batch` for free. Run top to bottom.

## 0. Setup

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join("..", ".env"))
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

from langchain.chat_models import init_chat_model
model = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.7)

print("Key loaded:", bool(os.getenv("GOOGLE_API_KEY")))

Key loaded: True


## 1. The manual way (what we're replacing)

Here's the two-step dance you already know: fill the template, invoke the model, grab `.text`. Nothing wrong with it — just repetitive once pipelines grow.

In [2]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a witty copywriter."),
    ("human", "Write a one-line slogan for a product that does: {product}"),
])

filled = prompt.invoke({"product": "a coffee mug that keeps drinks hot for 6 hours"})
response = model.invoke(filled)
print(response.text)

Here are a few witty one-liners for a coffee mug that keeps drinks hot for 6 hours:

*   Still steaming, six hours later.
*   Your last sip, as hot as your first. Six hours later.
*   Six hours hot. Your lukewarm days are over.
*   This mug's got staying power. Like, six hours of it.
*   Keeps your brew hot for longer than your morning meeting.


## 2. The same thing as a chain

Now connect the pieces with `|`. We also add a **`StrOutputParser`** at the end: it pulls the plain string out of the model's message, so the chain returns a `str` directly instead of an `AIMessage`.

One object, `chain`, holds the whole flow. You `.invoke` it with the template's variables.

In [3]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | model | StrOutputParser()

result = chain.invoke({"product": "a coffee mug that keeps drinks hot for 6 hours"})
print(type(result).__name__, "->", result)

TextAccessor -> Here are a few witty one-line slogans for a coffee mug that keeps drinks hot for 6 hours:

*   Your coffee's got stamina.
*   Six hours of hot coffee. Your microwave can relax.
*   Still hot. Still perfect. Six hours later.
*   Finally, a mug that understands "hot" means *hot*.
*   Never a lukewarm sip again.


## 3. The chain is reusable: invoke / stream / batch

Because the chain is a Runnable (just like a model in Module 01), it gets all three calling styles **for the entire pipeline** with no extra work.

In [4]:
# stream the whole prompt->model->parser pipeline:
print("STREAM: ", end="")
for piece in chain.stream({"product": "noise-cancelling headphones for cats"}):
    print(piece, end="", flush=True)

# batch many inputs at once:
print("BATCH:")
for slogan in chain.batch([
    {"product": "a self-watering plant pot"},
    {"product": "an umbrella that predicts rain"},
]):
    print("-", slogan)

STREAM: Here are a few witty one-line slogans for noise-cancelling headphones for cats:

**Short & Punchy:**

*   Silence the vacuum, save the cat.
*   Give your cat the silent treatment it deserves.
*   Quiet the world, unleash the purr.
*   Purr-fectly oblivious to your loud life.
*   For the cat who demands purr-fection in silence.

**A Bit More Descriptive/Attitude:**

*   Because even royalty needs quiet.
*   Muffle the madness, unleash the calm.
*   Finally, a peaceful nap is within paw's reach.
*   Tune out the chaos, tune into cat naps.BATCH:
- Here are a few one-line slogans for a self-watering plant pot, playing with different angles:

**Short & Sweet:**

*   Worry less, grow more.
*   Your plants, on autopilot.
*   Hydration, perfected.

**Witty & Clever:**

*   The secret to a green thumb (that isn't yours).
*   Finally, a plant that pulls its own weight.
*   Less H2-Oh no, more H2-Grow!
*   Give your plants independence.

**Benefit-Focused:**

*   Never forget to water aga

## 4. Add your own step with `RunnableLambda`

A chain step doesn't have to be a prompt or model — it can be **any function**. Wrap it with `RunnableLambda` (or just drop a plain lambda into the pipe; LCEL wraps it for you). This lets you post-process the model's output inside the chain.

Here we shout the slogan by upper-casing it as the final step.

In [5]:
from langchain_core.runnables import RunnableLambda

shout = RunnableLambda(lambda s: s.upper() + " 🔥")

loud_chain = prompt | model | StrOutputParser() | shout

print(loud_chain.invoke({"product": "a backpack with a built-in solar charger"}))

HERE ARE A FEW WITTY ONE-LINERS FOR A SOLAR-CHARGING BACKPACK:

*   NEVER RUN OUT OF JUICE, JUST ADD SUN.
*   YOUR ADVENTURE, FULLY CHARGED BY THE SKY.
*   THE ONLY PLUG YOU'LL NEED IS THE SUN.
*   CARRY THE SUN, POWER YOUR WORLD.
*   FINALLY, A BACKPACK THAT *GIVES* YOU ENERGY. 🔥


## 5. Compose chains — output of one into the next

Real pipelines have multiple model calls. With LCEL you pipe one chain's result into another. Below: chain A invents a **company name** from a product; we wrap that string into the input the second prompt expects, then chain B writes a **slogan** for that name.

`first_chain | (reshape) | second_chain` — the whole multi-step flow is again one object.

In [6]:
name_prompt = ChatPromptTemplate.from_messages([
    ("human", "Invent a single short brand name for a company that makes: {product}. Reply with only the name."),
])
slogan_prompt = ChatPromptTemplate.from_messages([
    ("human", "Write a 5-word slogan for a brand called '{company}'. Reply with only the slogan."),
])

name_chain = name_prompt | model | StrOutputParser()
slogan_chain = slogan_prompt | model | StrOutputParser()

# name_chain outputs a string -> reshape it into {"company": ...} -> slogan_chain
pipeline = name_chain | (lambda name: {"company": name.strip()}) | slogan_chain

product = "eco-friendly water bottles"
print("Slogan:", pipeline.invoke({"product": product}))

Slogan: Pure Goodness, Pure Joy, Always.


## Recap

- **LCEL** connects steps with `|`: `prompt | model | parser` is one **chain**.
- **`StrOutputParser`** turns the final `AIMessage` into a plain string.
- A chain is a Runnable, so **`invoke` / `stream` / `batch`** work on the whole pipeline for free.
- **`RunnableLambda`** (or a bare lambda in the pipe) inserts your own function as a step.
- **Compose chains** by piping one's output into the next — reshape between them as needed.

## 🛠️ Try it yourself

1. Build a `prompt | model | StrOutputParser()` chain for a task of your own and `invoke` it.
2. `stream` your chain and watch it type out.
3. `batch` your chain over three different inputs in one call.
4. Add a `RunnableLambda` step that trims whitespace and adds a prefix like `"Slogan: "`.
5. **Stretch:** make a 3-step pipeline (e.g. product → brand name → slogan → translate the slogan to French) by piping a third chain on the end.

When you're done, say **"next"** and we'll build **Module 07 — Memory**.